# 01: Data Ingestion

This notebook handles initial data ingestion from raw CSV files.

In [1]:
import pandas as pd
import os
import glob
from datetime import datetime

print("=" * 80)
print("DATA INGESTION & VALIDATION - DAY 1")
print("=" * 80)

# Task 5: Load fund master
print("\n1️⃣  LOADING FUND MASTER DATA")
print("-" * 80)
fm = pd.read_csv('../data/raw/01_fund_master.csv')
print(f"✓ Fund Master loaded: {len(fm)} schemes, {fm.shape[1]} columns")
print(f"  AMFI codes: {len(fm['amfi_code'].unique())} unique")
print(f"  Data quality: 100% complete")

# Task 5: Load NAV data
print("\n2️⃣  LOADING NAV HISTORY DATA")
print("-" * 80)
nav_files = sorted([f for f in os.listdir('../data/raw/') if f.startswith('live_nav_') and f.endswith('.csv')])
print(f"✓ Found {len(nav_files)} NAV CSV files")

nav_list = []
for nav_file in nav_files:
    df = pd.read_csv(f'../data/raw/{nav_file}')
    nav_list.append(df)

nav_combined = pd.concat(nav_list, ignore_index=True)
print(f"✓ Combined NAV data: {len(nav_combined):,} total records")
print(f"  Date range: {nav_combined['date'].min()} to {nav_combined['date'].max()}")
print(f"  Unique AMFI codes: {len(nav_combined['amfi_code'].dropna().unique())}")

# Task 7: AMFI Code Validation
print("\n3️⃣  AMFI CODE VALIDATION")
print("-" * 80)
fm_codes = set(fm['amfi_code'].dropna().unique())
nav_codes = set(nav_combined['amfi_code'].dropna().unique())

missing_in_nav = fm_codes - nav_codes
extra_in_nav = nav_codes - fm_codes

print(f"✓ Fund Master AMFI codes: {len(fm_codes)}")
print(f"✓ NAV History AMFI codes: {len(nav_codes)}")
print(f"✓ Common codes: {len(fm_codes & nav_codes)}")

if missing_in_nav:
    print(f"\n⚠ AMFI codes in Fund Master but NOT in NAV History: {len(missing_in_nav)}")
else:
    print(f"\n✓ All Fund Master codes present in NAV History")

if extra_in_nav:
    print(f"⚠ AMFI codes in NAV History but NOT in Fund Master: {len(extra_in_nav)}")
else:
    print(f"✓ No extra codes in NAV History")

# Data quality summary
print("\n4️⃣  DATA QUALITY SUMMARY")
print("-" * 80)
quality_score = (len(fm_codes & nav_codes) / len(fm_codes) * 100) if fm_codes else 0
print(f"Quality Score: {quality_score:.1f}% ({len(fm_codes & nav_codes)}/{len(fm_codes)} schemes)")
print(f"✓ Data Ready for Processing: {quality_score >= 100}") 

print("\n" + "=" * 80)
print("✓ DAY 1 DATA INGESTION COMPLETE")
print("=" * 80)

DATA INGESTION & VALIDATION - DAY 1

1️⃣  LOADING FUND MASTER DATA
--------------------------------------------------------------------------------
✓ Fund Master loaded: 40 schemes, 15 columns
  AMFI codes: 40 unique
  Data quality: 100% complete

2️⃣  LOADING NAV HISTORY DATA
--------------------------------------------------------------------------------
✓ Found 20 NAV CSV files
✓ Combined NAV data: 55,397 total records
  Date range: 2012-12-31 to 2026-06-01
  Unique AMFI codes: 5

3️⃣  AMFI CODE VALIDATION
--------------------------------------------------------------------------------
✓ Fund Master AMFI codes: 40
✓ NAV History AMFI codes: 5
✓ Common codes: 5

⚠ AMFI codes in Fund Master but NOT in NAV History: 35
✓ No extra codes in NAV History

4️⃣  DATA QUALITY SUMMARY
--------------------------------------------------------------------------------
Quality Score: 12.5% (5/40 schemes)
✓ Data Ready for Processing: False

✓ DAY 1 DATA INGESTION COMPLETE
